# 階層ベイズMNL（対角共分散）による比較

対角共分散の階層ベイズMNL（HB-MNL）で camera 実データと PerSearch 疑似データを比較する。

記事の分析設計で採用している **bayesm::camera の生の符号化のまま**
（brand は4列ダミー・pixels/zoom/video/swivel/wifi は0/1・price は100ドル単位の
連続値、「どれも選ばない」は全列0の行として表現）に、対角共分散の階層構造
`beta_i ~ Normal(beta_bar, diag(sigma^2))` を適用する。

非中心化パラメータ化（`z ~ Normal(0,1)` を `sigma_beta` と組み合わせる）で収束を安定させる。
camera・PerSearch 両方に同一モデル・同一符号化を適用し、母集団平均 `beta_bar` と
個人間標準偏差 `sigma_beta` を比較する。

**使い方**: 下の「実行設定」セルで `DATASET` を `"camera"` または `"persearch"` に
切り替えてから、最後の実行セルを走らせる（1回の実行で1データセット分）。


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt


In [ ]:
RESULTS_DIR = Path.cwd().parent / "results"

CAMERA_LONG_CSV = RESULTS_DIR / "camera_long.csv"
CAMERA_RAW_LONG_CSV = RESULTS_DIR / "camera_raw_long.csv"
PERSEARCH_RAW_LONG_CSV = RESULTS_DIR / "persearch_camera_raw_long.csv"

# prepare_persearch_camera.ipynb の RAW_COLUMNS と同じ並び（notebook 間 import はしないため直接定義）
RAW_COLUMNS = [
    "canon", "sony", "nikon", "panasonic",
    "pixels", "zoom", "video", "swivel", "wifi",
    "price_usd100",
]

BRAND_LEVEL_ORDER = ["canon", "sony", "nikon", "panasonic"]
BINARY_LEVEL_ORDER = {
    "pixels": ["低画素", "高画素"],
    "zoom": ["標準ズーム", "高倍率ズーム"],
    "video": ["動画機能なし", "動画機能あり"],
    "swivel": ["回転式液晶なし", "回転式液晶あり"],
    "wifi": ["Wi-Fiなし", "Wi-Fiあり"],
}


## 関数定義

In [ ]:
def camera_long_to_raw(df: pd.DataFrame) -> pd.DataFrame:
    """`camera_long.csv`（カテゴリ文字列の符号化）を raw 0/1・ドル価格の符号化に変換する。"""
    out = df[["respondent_id", "task_index", "option_label", "is_none", "chosen"]].copy()
    is_product = df["is_none"] == 0

    for brand in BRAND_LEVEL_ORDER:
        out[brand] = ((df["brand"] == brand) & is_product).astype(float)

    for attr, (ref, treat) in BINARY_LEVEL_ORDER.items():
        out[attr] = (df[attr] == treat).astype(float)
        out.loc[~is_product, attr] = 0.0

    out["price_usd100"] = df["price_usd100"].fillna(0.0)
    out.loc[~is_product, "price_usd100"] = 0.0

    return out[["respondent_id", "task_index", "option_label", "is_none", "chosen"] + RAW_COLUMNS]


In [ ]:
def load_raw_long(label: str) -> pd.DataFrame:
    if label == "camera":
        if CAMERA_RAW_LONG_CSV.exists():
            return pd.read_csv(CAMERA_RAW_LONG_CSV)
        df = camera_long_to_raw(pd.read_csv(CAMERA_LONG_CSV))
        df.to_csv(CAMERA_RAW_LONG_CSV, index=False, encoding="utf-8-sig")
        return df
    return pd.read_csv(PERSEARCH_RAW_LONG_CSV)


In [ ]:
def build_design(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, list[int], list[int]]:
    """long DataFrame から (n_resp, n_task, n_alt, 10) の設計配列を作る。

    データセットごとに n_alt（camera=5, persearch=4）が異なる点に対応するため、
    実データから option_label のユニーク数を都度読み取る。
    """
    resp_ids = sorted(df["respondent_id"].unique().tolist())
    task_ids = sorted(df["task_index"].unique().tolist())
    n_resp, n_task = len(resp_ids), len(task_ids)
    resp_pos = {r: i for i, r in enumerate(resp_ids)}
    task_pos = {t: i for i, t in enumerate(task_ids)}

    option_labels = sorted(df["option_label"].unique().tolist(), key=lambda x: (x == "none", x))
    n_alt = len(option_labels)
    opt_pos = {o: i for i, o in enumerate(option_labels)}

    K = len(RAW_COLUMNS)
    X = np.zeros((n_resp, n_task, n_alt, K), dtype=np.float64)
    choice_idx = np.zeros((n_resp, n_task), dtype=np.int64)

    df_sorted = df.sort_values(["respondent_id", "task_index"])
    r_arr = df_sorted["respondent_id"].map(resp_pos).to_numpy()
    t_arr = df_sorted["task_index"].map(task_pos).to_numpy()
    a_arr = df_sorted["option_label"].map(opt_pos).to_numpy()
    feat_arr = df_sorted[RAW_COLUMNS].to_numpy(dtype=np.float64)
    chosen_arr = df_sorted["chosen"].to_numpy()

    X[r_arr, t_arr, a_arr, :] = feat_arr
    chosen_mask = chosen_arr == 1
    choice_idx[r_arr[chosen_mask], t_arr[chosen_mask]] = a_arr[chosen_mask]

    return X, choice_idx, resp_ids, task_ids


In [ ]:
def fit_hb_diag(
    X: np.ndarray,
    choice_idx: np.ndarray,
    feature_names: list[str] = RAW_COLUMNS,
    draws: int = 1000,
    tune: int = 1000,
    chains: int = 2,
    target_accept: float = 0.9,
    seed: int = 42,
) -> az.InferenceData:
    n_resp, n_task, n_alt, K = X.shape
    coords = {"resp": np.arange(n_resp), "param": feature_names, "task": np.arange(n_task)}
    with pm.Model(coords=coords) as model:
        X_data = pm.Data("X_data", X)

        beta_bar = pm.Normal("beta_bar", mu=0.0, sigma=5.0, dims="param")
        sigma_beta = pm.HalfNormal("sigma_beta", sigma=2.5, dims="param")

        z = pm.Normal("z", mu=0.0, sigma=1.0, dims=("resp", "param"))
        beta_i = pm.Deterministic("beta_i", beta_bar + z * sigma_beta, dims=("resp", "param"))

        V = pt.einsum("rtak,rk->rta", X_data, beta_i)
        pm.Categorical("choice", logit_p=V, observed=choice_idx, dims=("resp", "task"))

        idata = pm.sample(
            draws=draws, tune=tune, chains=chains,
            target_accept=target_accept, random_seed=seed,
        )
    return idata


In [ ]:
def summarize(idata: az.InferenceData, feature_names: list[str]) -> pd.DataFrame:
    bb = az.summary(idata, var_names=["beta_bar"], ci_prob=0.94, ci_kind="hdi", round_to="none")
    sb = az.summary(idata, var_names=["sigma_beta"], ci_prob=0.94, ci_kind="hdi", round_to="none")
    bb = bb.reset_index().rename(columns={"index": "param"})
    sb = sb.reset_index().rename(columns={"index": "param"})
    bb["param"] = feature_names
    sb["param"] = feature_names
    return bb.merge(sb, on="param", suffixes=("_beta_bar", "_sigma_beta"))


In [ ]:
def importance_from_summary(summary: pd.DataFrame, price_range: float) -> pd.DataFrame:
    coefs = dict(zip(summary["param"], summary["mean_beta_bar"]))
    raw_importance = {
        "brand": max(coefs[b] for b in BRAND_LEVEL_ORDER) - min(coefs[b] for b in BRAND_LEVEL_ORDER),
        "pixels": abs(coefs["pixels"]),
        "zoom": abs(coefs["zoom"]),
        "video": abs(coefs["video"]),
        "swivel": abs(coefs["swivel"]),
        "wifi": abs(coefs["wifi"]),
        "price_usd100": abs(coefs["price_usd100"]) * price_range,
    }
    total = sum(raw_importance.values())
    imp = pd.Series(raw_importance) / total
    return imp.sort_values(ascending=False).rename("importance").reset_index().rename(columns={"index": "attr"})


In [ ]:
def run(
    label: str,
    draws: int = 1000,
    tune: int = 1000,
    chains: int = 2,
    target_accept: float = 0.9,
    n_resp_limit: int | None = None,
) -> None:
    df = load_raw_long(label)
    if n_resp_limit is not None:
        keep_ids = sorted(df["respondent_id"].unique())[:n_resp_limit]
        df = df[df["respondent_id"].isin(keep_ids)]

    price_vals = df.loc[df["is_none"] == 0, "price_usd100"]
    price_range = float(price_vals.max() - price_vals.min())

    t0 = time.time()
    X, choice_idx, resp_ids, task_ids = build_design(df)
    print(f"[{label}] design matrix: X.shape={X.shape}, build={time.time() - t0:.1f}s")

    t0 = time.time()
    idata = fit_hb_diag(X, choice_idx, draws=draws, tune=tune, chains=chains, target_accept=target_accept)
    elapsed = time.time() - t0
    print(f"[{label}] sampling done in {elapsed:.1f}s")

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    idata.to_netcdf(RESULTS_DIR / f"hb_diag_{label}_trace.nc")

    summary = summarize(idata, RAW_COLUMNS)
    summary["elapsed_sec"] = elapsed
    summary.to_csv(RESULTS_DIR / f"hb_diag_{label}_summary.csv", index=False, encoding="utf-8-sig")

    imp = importance_from_summary(summary, price_range)
    imp.to_csv(RESULTS_DIR / f"hb_diag_{label}_importance.csv", index=False, encoding="utf-8-sig")

    print(summary[["param", "mean_beta_bar", "r_hat_beta_bar", "ess_bulk_beta_bar", "mean_sigma_beta", "r_hat_sigma_beta"]].to_string(index=False))
    print(imp.to_string(index=False))
    print(f"max r_hat (beta_bar): {summary['r_hat_beta_bar'].max()}, min ess_bulk: {summary['ess_bulk_beta_bar'].min()}")


## 実行設定

`DATASET` を書き換えてから実行セルを走らせる。`camera` は約2.4分、`persearch`（camera と同じ332体構成）は約4.7分
（12コア環境、C コンパイラ未導入・pytensor の numpy フォールバックのままの実測値。
課題数・収束の難しさにも左右されるため、目安として捉えること）。

In [ ]:
DATASET = "camera"  # "camera" または "persearch"
DRAWS = 1000
TUNE = 1000
CHAINS = 2
TARGET_ACCEPT = 0.9
N_RESP_LIMIT = None  # 動作確認だけしたい場合は 20 などに絞る


## 実行

`r_hat` が1.01未満、`ess_bulk` が400超であることが理想。回答者数を camera と揃えた
現在のデータでは chains=2・draws=1000 でも camera 側 r_hat=1.005/ess_bulk=314、
persearch 側 r_hat=1.011/ess_bulk=169 まで収束する（回答者数が少ない条件ではもっと
不安定になりやすいので注意）。収束を上げたい場合は `draws`・`chains` を増やす（chains=4 が既定の推奨値）。

In [ ]:
run(
    DATASET,
    draws=DRAWS, tune=TUNE, chains=CHAINS,
    target_accept=TARGET_ACCEPT, n_resp_limit=N_RESP_LIMIT,
)
